# Module 11 — File Handling

**Course:** Programming Fundamentals Using Python

**Navigation:** [01 Introduction](01_Introduction_to_Programming_and_Python.ipynb) | [02 Variables](02_Variables_Data_Types_and_Operators.ipynb) | [03 Input/Output](03_Input_and_Output.ipynb) | [04 Strings](04_Strings.ipynb) | [05 Decisions](05_Decision_Making.ipynb) | [06 Loops](06_Loops_and_Iteration.ipynb) | [06b Mini Project](06b_Mini_Project_Quiz_and_Grade_Report.ipynb) | [07 Lists](07_Lists.ipynb) | [08 Tuples/Sets/Dicts](08_Tuples_Sets_and_Dictionaries.ipynb) | [09 Functions](09_Functions.ipynb) | [10 Modules](10_Modules_and_Packages.ipynb) | **[11 Files](11_File_Handling.ipynb)** | [12 Exceptions](12_Exception_Handling.ipynb) | [13 Problem Solving](13_Problem_Solving_and_Algorithmic_Thinking.ipynb) | [14 OOP](14_Introduction_to_OOP.ipynb) | [15 Data Handling](15_Python_Data_Handling.ipynb) | [16 Computational Thinking](16_Computational_Thinking.ipynb)

---

## Module overview

Variables die when the program ends. **Files** make data *persistent*:
written once, readable forever, by any program. This module covers text
files end to end — opening modes, reading, writing, the `with` statement,
paths, CSV — and builds a file-based student record system.

## Learning objectives

By the end of this module you can:

* explain why files matter and what persistence means
* choose the right mode: `r`, `w`, `a`, `x` (and recognise `b`)
* read files with `read()`, `readline()`, `readlines()` and line-by-line loops
* write with `write()` and `writelines()`
* always use `with open(...) as file:` and explain why
* handle relative vs absolute paths and use `pathlib.Path`
* read and write CSV files with the `csv` module (plain and `Dict` variants)
* recognise the classic file errors: `FileNotFoundError`, `PermissionError`

## Prerequisites

Modules 1–10 (strings, loops, lists, dictionaries, functions, modules).

**Sandbox note:** every file this notebook creates goes into a single
`file_sandbox` folder, created and cleaned below — no clutter, no
accidental overwrites outside the notebook.

In [ ]:
from pathlib import Path
import shutil

SANDBOX = Path("file_sandbox")
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)        # fresh sandbox on every full run
SANDBOX.mkdir()
print("Sandbox ready:", SANDBOX)

## Contents

1. Why file handling? Files vs variables
2. Opening files and the modes `r w a x b`
3. Reading: `read()`, `readline()`, `readlines()`, line by line
4. Writing: `write()`, `writelines()`
5. Closing files — and the `with` statement
6. Practical text files: notes, logs, to-do lists, reports
7. File paths: relative, absolute, `pathlib`
8. CSV with the standard library
9. Common file errors
10. Mini project: File-Based Student Record System
11. Common mistakes
12. Lab 11 — Files in practice
13. Exercises
14. Summary and final assessment

## 1. Why file handling?

### Concept

A **file** is a named collection of bytes on disk. A **text file** stores
human-readable characters in lines; that is what we use here (Module 15
adds structured text: CSV and JSON).

### Why do we need it?

* **Persistence** — a quiz score in a variable vanishes; a score in a file
  survives to tomorrow's class.
* **Volume** — a list of 10 students fits in code; 10,000 students must
  live *outside* the program.
* **Sharing** — files move between programs, computers, people (you email
  a `.csv`, not a Python list).

**Files vs variables:** variables are fast but temporary and small; files
are slower but permanent and arbitrarily large. Real programs combine
both: load → process in memory → save.

## 2. Opening files and the modes

### Concept

`open(path, mode)` connects your program to a file and returns a **file
object**. The mode says what you intend to do:

| Mode | Meaning | If file exists | If file missing |
|---|---|---|---|
| `"r"` | read (default) | opens at the start | **FileNotFoundError** |
| `"w"` | write | **ERASES contents!** | creates it |
| `"a"` | append | writes at the end | creates it |
| `"x"` | exclusive create | **FileExistsError** | creates it |
| `"rb"`/`"wb"` | read/write **b**ytes | — | — |

⚠️ **The `w` trap:** opening with `"w"` immediately empties an existing
file — *before* you write anything. There is no undo. Use `"a"` to add to
logs; use `"x"` when a file must not be clobbered.

### Syntax

```python
file = open("notes.txt", "r")   # old style — see `with` below
...
file.close()
```

In [ ]:
# Create a small file to read (writing is explained in section 4):
notes_path = SANDBOX / "notes.txt"
notes_path.write_text("first line\nsecond line\nthird line\n", encoding="utf-8")

file = open(notes_path, "r")
content = file.read()
file.close()
print(content)

## 3. Reading files — four ways

### Concept

| Call | Returns | Use when |
|---|---|---|
| `read()` | the WHOLE text as one string | small files, quick peek |
| `readline()` | one line (with its `\n`) | you stop early / custom loop |
| `readlines()` | a LIST of all lines | you want list indexing |
| `for line in file:` | one line per iteration | **the idiomatic default** |

### Example 1 — Beginner: the four compared

In [ ]:
notes_path = SANDBOX / "notes.txt"

# 1) read(): everything, one string
file = open(notes_path, "r")
whole = file.read()
print("read():", len(whole), "characters")   # 11+1 + 11+1 + 10+1 = 34
file.close()

# 2) readlines(): list of lines
file = open(notes_path, "r")
lines = file.readlines()
print("readlines():", lines)
file.close()

# 3) readline(): just the first line
file = open(notes_path, "r")
first = file.readline()
print("readline():", first, end="")     # line already contains its \n
file.close()

# 4) the loop: one line at a time (best for big files)
file = open(notes_path, "r")
for line in file:
    print("loop saw:", line, end="")
file.close()

### Explanation

* Each line read keeps its trailing newline `\n` — that's why `end=""`
  prevents double spacing in prints.
* `.strip()` (Module 4) removes the newline when you don't want it.

### Example 2 — Practical: processing lines

In [ ]:
notes_path = SANDBOX / "notes.txt"

file = open(notes_path, "r")
line_number = 1
for line in file:
    clean = line.strip()                 # drop the \n
    print(f"{line_number}: {clean.upper()}")
    line_number += 1
file.close()

## 4. Writing files

### Concept

* `file.write(text)` writes exactly the string given — **no newline is
  added**, you add `\n` yourself.
* `file.writelines(list_of_strings)` writes them all, also without
  adding newlines.
* Both need mode `"w"` (fresh) or `"a"` (append).

In [ ]:
fruits_path = SANDBOX / "fruits.txt"

file = open(fruits_path, "w")            # creates/overwrites
file.write("apple\n")
file.write("banana\n")
file.write("mango\n")
file.close()

print(open(fruits_path).read())          # quick peek (see section 5!)

In [ ]:
fruit_list = ["kiwi\n", "plum\n", "fig\n"]
file = open(fruits_path, "a")            # APPEND — keeps apple/banana/mango
file.writelines(fruit_list)
file.close()

print(open(fruits_path).read())

### Common Mistakes

```python
# 1. write(42)         # TypeError — write() needs str; use str(42) or f-strings
# 2. Forgetting \n:     all rows end up on ONE line
# 3. Opening for read a file you meant to create:  FileNotFoundError
# 4. The w trap: reopening a log with "w" wipes yesterday's log.
```

## 5. Closing files — and why `with` is the answer

### Concept

Files must be **closed** to flush pending writes and release the operating
system's lock. Forgetting to close risks **lost data** (the last buffer
never reaches disk) and locked files. But manual `close()` calls get
skipped whenever an error happens halfway — so Python has the `with`
statement: it closes the file **even if an error occurs**.

### Syntax (the professional default)

```python
with open(path, "r") as file:
    ... use file ...
# file is closed automatically here, error or not
```

In [ ]:
poem_path = SANDBOX / "poem.txt"

with open(poem_path, "w") as file:
    file.write("Roses are red\n")
    file.write("Violets are blue\n")
# outside the with-block: already closed and flushed

with open(poem_path, "r") as file:
    print(file.read())

From here on, **every** example uses `with` — make it your reflex.

## 6. Practical text files

### Example 1 — A simple logger (append mode)

In [ ]:
log_path = SANDBOX / "app.log"

with open(log_path, "a") as log_file:        # append: history is preserved
    log_file.write("2026-09-12 09:00 app started\n")
    log_file.write("2026-09-12 09:01 user logged in\n")
    log_file.write("2026-09-12 09:05 data saved\n")

with open(log_path, "r") as log_file:
    for line in log_file:
        print(line, end="")

### Example 2 — A to-do list file that survives restarts

In [ ]:
todo_path = SANDBOX / "todo.txt"

# Write today's tasks:
with open(todo_path, "w") as file:
    file.write("finish module 11 exercises\n")
    file.write("review slicing rules\n")
    file.write("sleep\n")

# Later — read, process, report:
with open(todo_path, "r") as file:
    tasks = [line.strip() for line in file]     # list comprehension light;
                                                # a plain for+append loop works too
print(len(tasks), "tasks:")
position = 1
for task in tasks:
    print(f"{position}. {task}")
    position += 1

### Example 3 — A formatted report from data

In [ ]:
report_path = SANDBOX / "report.txt"
marks = [("Ali", 78), ("Sara", 91), ("Omar", 84)]

with open(report_path, "w") as file:
    file.write(f"{'Student':<10}{'Marks':>6}\n")
    file.write("-" * 16 + "\n")
    for name, score in marks:
        file.write(f"{name:<10}{score:>6}\n")
    file.write("-" * 16 + "\n")
    average = sum(score for _, score in marks) / len(marks)   # see note below
    file.write(f"Class average: {average:.1f}\n")

print(open(report_path).read())

(The `sum(score for _, score in marks)` line is a *generator expression* —
a Module 16 preview. The fully-taught way is a loop:
`total = 0` / `for name, score in marks: total += score` /
`average = total / len(marks)`.)

## 7. File paths — relative, absolute, pathlib

### Concept

* A **relative path** ("`data/students.csv`") is read from the **current
  working directory** — for Jupyter, the folder where the notebook lives.
* An **absolute path** ("/home/you/data/students.csv",
  "C:\\Users\\you\\data\\students.csv") works from anywhere but ties your
  program to one machine.
* **`pathlib.Path`** (standard library) builds paths safely across
  Windows/macOS/Linux: `Path("data") / "students.csv"` — no more
  slash-vs-backslash bugs.

In [ ]:
from pathlib import Path

print("parts:", (Path("file_sandbox") / "notes.txt").as_posix())

notes = Path("file_sandbox") / "notes.txt"
print("exists:", notes.exists())
print("is a file:", notes.is_file())

(Your `cwd:` line shows *your* folder — machine-specific by nature. Note
also `Path` prints with `/` or `\` depending on the OS: both are correct.)

### Example — Practical: safe path handling

In [ ]:
from pathlib import Path

data_dir = Path("file_sandbox")
data_dir.mkdir(exist_ok=True)          # folder ready if missing

target = data_dir / "welcome.txt"
with open(target, "w") as file:
    file.write("Paths built with pathlib work everywhere.\n")

with open(target, "r") as file:
    print(file.read().strip())

## 8. CSV — the universal data format

### Concept

A **CSV** (comma-separated values) file stores a table as text: one row per
line, values separated by commas, first row usually the headers. The
standard library's `csv` module parses it correctly (quotes, commas inside
values, ...).

```text
name,marks,grade
Ali,78,B
Sara,91,A
```

### Example 1 — Writing and reading with `csv.writer` / `csv.reader`

In [ ]:
import csv

csv_path = SANDBOX / "students.csv"

# --- write ---
with open(csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["name", "marks", "grade"])    # header row
    writer.writerow(["Ali", 78, "B"])
    writer.writerow(["Sara", 91, "A"])
    writer.writerow(["Omar", 84, "B"])

# --- read ---
with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    rows = list(reader)                 # every row becomes a list of strings

for row in rows:
    print(row)

**Two crucial details:**

* `newline=""` is the officially recommended setting for CSV files — it
  prevents blank lines between rows on Windows.
* **Everything reads back as strings** (`"78"`, not `78`) — convert with
  `int()`/`float()` when you need numbers (Module 15's cleaning section
  builds a habit around this).

### Example 2 — Processing the rows

In [ ]:
import csv

csv_path = SANDBOX / "students.csv"

with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    header = next(reader)               # consume the header row
    print("Header:", header)

    total = 0
    count = 0
    for row in reader:
        total += int(row[1])            # convert the marks column
        count += 1
        print(row[0], "scored", row[1])

print(f"Average marks: {total / count:.1f}")

### Example 3 — `DictReader` and `DictWriter`: rows as dictionaries

Column *names* beat column *numbers* — `DictReader` hands each row as a
dictionary keyed by the header:

In [ ]:
import csv

csv_path = SANDBOX / "students.csv"

with open(csv_path, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:                  # row is a dict: {"name": "Ali", ...}
        print(row["name"], "->", row["grade"], "-", row["marks"])

In [ ]:
import csv

out_path = SANDBOX / "products.csv"

with open(out_path, "w", newline="", encoding="utf-8") as file:
    fieldnames = ["product", "price", "stock"]
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerow({"product": "pen", "price": 30, "stock": 120})
    writer.writerow({"product": "notebook", "price": 250, "stock": 45})

with open(out_path, "r", newline="", encoding="utf-8") as file:
    for row in csv.DictReader(file):
        print(f"{row['product']:<9} Rs {row['price']:>4}  x{row['stock']}")

## 9. Common file errors

### Concept

* **`FileNotFoundError`** — mode `r` on a missing file (or a typo/path bug).
* **`PermissionError`** — the OS refuses access (locked, read-only, protected folder).
* **Wrong mode** — `write()` on an `r`-handle raises
  `io.UnsupportedOperation`; reading a `w`-handle the same.
* **Encoding** — text files are bytes + an encoding; sticking to
  `encoding="utf-8"` on both ends avoids most "weird characters" bugs.

Module 12 teaches `try/except` to *handle* these; for now, learn to
*recognise* them:

In [ ]:
# A controlled look at the error (this cell deliberately catches it):
missing = SANDBOX / "definitely_not_here.txt"

try:
    file = open(missing, "r")
    file.close()
except FileNotFoundError:
    print("FileNotFoundError: the file does not exist (yet).")

(That `try/except` is a full preview of Module 12 — a taste, not the
course. The mode-mismatch error you can try in a scratch cell:
`f = open(notes_path); f.write("x")` → `io.UnsupportedOperation: not
writable`.)

## 10. Mini project — File-Based Student Record System

**Requirements**

1. Menu: add / list / search / save / load / quit.
2. Students live in memory as a list of dictionaries.
3. `save` writes a CSV; `load` reads it back (replacing memory).
4. `search` is case-insensitive on the name.
5. All file access uses `with` and `pathlib` paths.

**Pseudocode**

```text
students <- []
LOOP:
    choice <- menu
    add:    name, marks <- ask; students.append({name, marks})
    list:   FOR student IN students: print row
    search: FOR student IN students: IF name matches: print
    save:   csv.writer over students.csv (header + rows)
    load:   rows <- csv.DictReader; rebuild students list
    quit:   break
```

In [ ]:
# ----- File-Based Student Record System -----
import csv
from pathlib import Path

RECORDS = Path("file_sandbox") / "records.csv"
students = []

while True:
    print()
    print("--- Student Records ---")
    print("add / list / search / save / load / quit")
    choice = input("Choice: ").strip().lower()

    if choice == "add":
        name = input("Name: ").strip().title()
        marks = float(input("Marks: "))
        students.append({"name": name, "marks": marks})
        print(f"Added {name}.")
    elif choice == "list":
        if len(students) == 0:
            print("(no students in memory)")
        for student in students:
            print(f"{student['name']:<10} {student['marks']:.0f}")
    elif choice == "search":
        query = input("Name to find: ").strip().lower()
        found = False
        for student in students:
            if student["name"].lower() == query:
                print(f"Found: {student['name']} with {student['marks']:.0f} marks")
                found = True
        if not found:
            print("No match.")
    elif choice == "save":
        RECORDS.parent.mkdir(exist_ok=True)
        with open(RECORDS, "w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["name", "marks"])
            for student in students:
                writer.writerow([student["name"], student["marks"]])
        print(f"Saved {len(students)} student(s) to {RECORDS}.")
    elif choice == "load":
        students = []                      # replace memory with the file
        with open(RECORDS, "r", newline="", encoding="utf-8") as file:
            for row in csv.DictReader(file):
                students.append({"name": row["name"], "marks": float(row["marks"])})
        print(f"Loaded {len(students)} student(s).")
    elif choice == "quit":
        print("Goodbye!")
        break
    else:
        print("Unknown choice.")

### Sample run (the session used to validate this notebook)

```text
add ali / add sara   -> Added Ali. / Added Sara.
list                 -> Ali 78 / Sara 91
search SARA          -> Found: Sara with 91 marks
save                 -> Saved 2 student(s) to file_sandbox/records.csv.
add omar             -> Added Omar.          (memory now has 3)
load                 -> Loaded 2 student(s). (file replaces memory!)
list                 -> Ali 78 / Sara 91     (Omar was never saved)
quit                 -> Goodbye!
```

The Omar lesson is deliberate: **unsaved data is not data.** In the lab,
improve this: auto-save after every add, or a `delete` command, or marks
validation.

## 11. Common mistakes — the module in one list

```python
# 1. Reopening with "w" when "a" was meant     -> yesterday's log, gone
# 2. Forgetting newline="" for CSV on Windows -> blank rows everywhere
# 3. Forgetting \n in write()                  -> one endless line
# 4. Manual open/close without with            -> leaks on error paths
# 5. Comparing numbers read from CSV as text   -> "9" > "10" is True!
# 6. Typoed relative paths                     -> FileNotFoundError
```

## 12. Lab 11 — Files in practice

### Learning objectives

Create, read, append and process text and CSV files confidently; build
paths with pathlib; develop the save-your-work reflex.

### Short theory recap

Modes: `r` read, `w` overwrite, `a` append, `x` exclusive-create, `b`
bytes. `with open(...) as f:` closes for you. CSV: `csv.writer`/`reader`
for lists, `DictWriter`/`DictReader` for dictionaries; everything textual
reads back as `str`.

### Warm-up (run and predict)

1. Open a sandbox file with `"x"` twice — what happens the second time?
2. `open(path, "w")` then immediately `close()` — what is in the file?
3. Reading a file line by line, what does each `line` end with?

### Guided exercises

In [ ]:
# Guided 1 — word, line and character census of a text file.
target = SANDBOX / "census.txt"
target.write_text("python is fun\nfiles are power\npersist everything\n", encoding="utf-8")

lines = 0
words = 0
characters = 0
with open(target, "r") as file:
    for line in file:
        lines += 1
        characters += len(line)
        words += len(line.split())

print(f"lines={lines} words={words} characters={characters}")

In [ ]:
# Guided 2 — guest book: append three visitors, then display all with numbers.
guest_path = SANDBOX / "guestbook.txt"

visitors = ["Amina", "Bilal", "Sara"]
with open(guest_path, "a") as file:
    for visitor in visitors:
        file.write(visitor + "\n")

with open(guest_path, "r") as file:
    number = 1
    for line in file:
        print(f"{number}. {line.strip()}")
        number += 1

In [ ]:
# Guided 3 — CSV round trip with DictWriter/DictReader.
import csv

inventory_path = SANDBOX / "inventory.csv"
items = [
    {"item": "keyboard", "qty": 12},
    {"item": "mouse", "qty": 30},
]

with open(inventory_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["item", "qty"])
    writer.writeheader()
    for item in items:
        writer.writerow(item)

with open(inventory_path, "r", newline="", encoding="utf-8") as file:
    total_qty = 0
    for row in csv.DictReader(file):
        total_qty += int(row["qty"])       # str -> int conversion!
        print(row["item"], "x", row["qty"])
print("Total quantity:", total_qty)

### Independent exercises

1. Write your weekly plan to `plan.txt` (one task per line); read it back
   numbered.
2. Append three more tasks to `plan.txt` and prove the old ones survived.
3. Read `plan.txt` and print only tasks containing the letter "e".
4. Save `[[1, 2], [3, 4]]` to a CSV; read it back and print the sum of
   each row.
5. Build `grades.csv` with DictWriter (name, quiz, final); read with
   DictReader and print each student's total (convert to int!).
6. Count how many lines of `fruits.txt` start with "a".
7. Create `empty.txt` via `"x"` mode; confirm `exists()`; then try `"x"`
   again and read the error name.
8. Write a file whose lines are `name,city` for 3 friends; read it and
   print `city — name` sorted by city.
9. Use `Path("file_sandbox") / "notes.txt"` to print the file's size
   before and after appending a line.
10. Copy a file's content to a second file (read one, write the other).

### Challenge problems

1. **Log analyser.** The `app.log` from section 6: count lines, and print
    the most recent entry (last line) without loading the whole file into
    a list (read line by line, keep updating `last_line`).
2. **CSV merger.** Two CSVs with the same header; write a third containing
    all rows of both (headers written once).
3. **Simple database.** A menu program over `contacts.csv`: add (append
    row), list (read all), and delete-by-name (read all, rewrite the file
    without the match — the classic read-modify-write cycle).

## 13. Exercises

## Quick Review

1. Which mode erases the file the moment it opens?
2. What does `readlines()` return that `read()` does not?
3. Why add `\n` manually with `write()`?
4. Name two things `with` guarantees that manual `close()` does not.
5. What is a relative path relative *to*?
6. Why prefer `pathlib` over glued strings like "data" + "/" + name?
7. What does `next(reader)` do on a `csv.reader`?
8. What type is every value read from a CSV file?
9. Which mode would you use for an audit log, and why?
10. What exception does `open` raise for a missing file in `r` mode?

<details><summary>Answers</summary>

1. `"w"`.
2. A list of lines (each keeping its `\n`); `read()` gives one string.
3. `write()` writes exactly what you pass — no automatic newline.
4. Closing even when an exception occurs; immediate flush to disk.
5. The current working directory (for notebooks: their own folder).
6. `Path` joins with the OS-correct separator and stays cross-platform.
7. Consumes and returns the next row — used to skip the header.
8. `str` — convert numbers explicitly.
9. `"a"` — history must never be overwritten.
10. `FileNotFoundError`.

</details>

## Predict the Output

In [ ]:
# Snippet 1 — prediction:
p = SANDBOX / "snip1.txt"
with open(p, "w") as f:
    f.write("a")
    f.write("b")
    f.write("c")
print(open(p).read())

In [ ]:
# Snippet 2 — prediction:
p = SANDBOX / "snip2.txt"
with open(p, "w") as f:
    f.write("1\n2\n")
with open(p, "r") as f:
    lines = f.readlines()
print(len(lines), lines[0].strip())

In [ ]:
# Snippet 3 — prediction:
p = SANDBOX / "snip3.txt"
with open(p, "w") as f:
    f.write("x\n")
with open(p, "a") as f:
    f.write("y\n")
print(open(p).read().strip().splitlines())

In [ ]:
# Snippet 4 — prediction:
import csv
p = SANDBOX / "snip4.csv"
with open(p, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["a", "b"])
    w.writerow([1, 2])
with open(p, "r", newline="") as f:
    rows = list(csv.reader(f))
print(rows[1][0] + rows[1][1])

In [ ]:
# Snippet 5 — prediction:
from pathlib import Path
p = Path("file_sandbox") / "snip5.txt"
with open(p, "w") as f:
    f.write("hello")
print(p.exists(), len(open(p).read()))

<details><summary>Answers</summary>

1. `abc` — writes concatenate with no separators.
2. `2 1` — two lines; first line's content is `1`.
3. `['x', 'y']` — append keeps existing content.
4. `12` — CSV values are strings; `+` concatenates "1" and "2".
5. `True 5` — the file exists and holds 5 characters.

</details>

## Find the Error

**Error 1**

```python
file = open("data.txt")
print(file.read())
# program ends — file never closed
```

**Error 2**

```python
with open("log.txt", "w") as f:
    f.write("new entry\n")
# ...meant to KEEP yesterday's log
```

**Error 3**

```python
with open("notes.txt", "r") as f:
    f.write("hello")
```

**Error 4**

```python
import csv
with open("students.csv", newline="") as f:
    for row in csv.DictReader(f):
        print(row["marks"] + 1)      # marks is "78"
```

**Error 5**

```python
with open("missing_file.txt") as f:
    print(f.read())
```

**Error 6**

```python
f = open("out.txt", "w")
f.write("data")
f = open("out.txt", "r")     # second open before closing the first
print(f.read())
```

<details><summary>Answers</summary>

1. No close → possible data loss/locks. Use `with open(...)`.
2. `"w"` wiped the log. Use `"a"`.
3. `io.UnsupportedOperation: not writable` — the handle is read-only.
4. `TypeError`: `row["marks"]` is a string. Use `int(row["marks"]) + 1`.
5. `FileNotFoundError` — the file does not exist.
6. The first handle is never closed (resource leak; on Windows the second
   open may even fail). Close before reopening — or better, two `with`
   blocks.

</details>

## Programming Exercises

1. **Diary.** Append a dated line to `diary.txt` (any date text); read the
   whole diary numbered.
2. **Word length report.** For a sentence, write each word and its length
   to `words.csv` (DictWriter); read back and print the longest word.
3. **Marks to grades file.** From a marks list, write `name,marks` CSV;
   read it and write a second CSV `name,grade` using Module 5 thresholds.
4. **Safe append.** Write a function `add_task(path, task)` that appends;
   call it three times; verify with a read.
5. **Line filter.** Copy only lines longer than 10 characters from one
   file to another.
6. **Character census.** For `census.txt`, count how many times the letter
   "e" appears (read line by line, `.count("e")` per line).
7. **Table writer.** Write a multiplication table (1–5) to `tables.txt`
   with aligned f-string columns.
8. **Path explorer.** Print `cwd`, whether `file_sandbox/notes.txt`
   exists, and its size — all via pathlib.
9. **CSV statistics.** From `students.csv`, compute and print the highest,
   lowest and average marks (numbers converted!).
10. **Backup habit.** Read any sandbox file and write a copy with suffix
    `_backup.txt`; verify both exist.

## Challenge Problems

1. **Unique names.** `names.txt` has duplicates; write `unique_names.txt`
    preserving first-seen order (sets for membership, list for order).
2. **Student record v2.** Extend the mini project: `delete` command
    (read → filter → rewrite) and an auto-load at startup if the CSV
    exists.
3. **Word frequency file.** Read a paragraph file; count word frequencies
    (Module 8 pattern); write `frequency.csv` (word,count) sorted by count
    descending — no imports beyond `csv`/`pathlib`.

## 14. Summary

* Files make data persistent; variables are fast but temporary.
* Modes: `r` read, `w` overwrite (dangerous!), `a` append, `x` create-only,
  `b` bytes.
* Reading: `read()`, `readline()`, `readlines()`, or the idiomatic
  line-by-line loop.
* Writing: `write()`/`writelines()` with manual `\n`; `with open(...)` is
  the default — it closes even on errors.
* Paths: relative to the working directory; build them with
  `pathlib.Path` for cross-platform safety.
* CSV: `csv.writer`/`reader` for lists, `DictWriter`/`DictReader` for
  dictionaries; `newline=""`; values come back as strings.
* Classic errors: `FileNotFoundError`, `PermissionError`, wrong mode,
  encoding surprises — recognised here, handled in Module 12.

## Final assessment (check yourself)

1. One line each: create `a.txt` containing "hi", then append "!".
2. A program crashes halfway through writing with a manual `open/close`.
    What may be lost, and how does `with` change that?
3. Why does `row["marks"] + 1` fail on freshly-read CSV data, and what is
    the two-character fix?
4. Write the pathlib expression for a file `q3.csv` inside a `data` folder
    inside the sandbox.

---

**Next up:** exception handling — what to do when things go wrong.

| Previous | Next |
|---|---|
| [Module 10 — Modules and Packages](10_Modules_and_Packages.ipynb) | [Module 12 — Exception Handling](12_Exception_Handling.ipynb) |